In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np

import nltk
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [3]:
path = "/content/drive/MyDrive/banking-ai-knowledge-system/dataset/banking_queries.csv"

df = pd.read_csv(path)

df.head()

,text,category
0,I am still waiting on my card?,card_arrival
1,What can I do if my card still hasn't arrived ...,card_arrival
2,I have been waiting over a week. Is the card s...,card_arrival
3,Can I track my card while it is in the process...,card_arrival
4,"How do I know if I will get my card, or if it ...",card_arrival


In [4]:
print("Dataset size:", df.shape)
print("Number of intents:", df["category"].nunique())

Dataset size: (10003, 2)


KeyError: 'intent'

In [5]:
print("Dataset size:", df.shape)
print("Number of category:", df["category"].nunique())

Dataset size: (10003, 2)
Number of category: 77


In [6]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [7]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

In [8]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r"[^\w\s]", "", text)

    words = text.split()

    words = [w for w in words if w not in stop_words]

    return " ".join(words)

In [9]:
df["clean_query"] = df["text"].apply(clean_text)

df.head()

KeyError: 'query'

In [10]:
df["clean_text"] = df["text"].apply(clean_text)

df.head()

,text,category,clean_text
0,I am still waiting on my card?,card_arrival,still waiting card
1,What can I do if my card still hasn't arrived ...,card_arrival,card still hasnt arrived 2 weeks
2,I have been waiting over a week. Is the card s...,card_arrival,waiting week card still coming
3,Can I track my card while it is in the process...,card_arrival,track card process delivery
4,"How do I know if I will get my card, or if it ...",card_arrival,know get card lost


In [11]:
X = df["clean_text"]
y = df["category"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [12]:
vectorizer = TfidfVectorizer()

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [13]:
model = LogisticRegression(max_iter=200)

model.fit(X_train_vec, y_train)

LogisticRegression(max_iter=200)

In [14]:
y_pred = model.predict(X_test_vec)

print(classification_report(y_test, y_pred))

                                                  precision    recall  f1-score   support

                           Refund_not_showing_up       0.88      0.94      0.91        32
                                activate_my_card       1.00      0.90      0.95        31
                                       age_limit       1.00      0.96      0.98        25
                         apple_pay_or_google_pay       1.00      1.00      1.00        23
                                     atm_support       1.00      0.84      0.91        19
                                automatic_top_up       1.00      0.96      0.98        27
         balance_not_updated_after_bank_transfer       0.78      0.81      0.79        31
balance_not_updated_after_cheque_or_cash_deposit       0.90      0.85      0.88        41
                         beneficiary_not_allowed       0.85      0.96      0.90        24
                                 cancel_transfer       0.89      0.94      0.92        35
         

In [15]:
def predict_intent(query):

    cleaned = clean_text(query)

    vec = vectorizer.transform([cleaned])

    pred = model.predict(vec)[0]

    return pred

In [16]:
predict_intent("My card was stolen")

'lost_or_stolen_card'

In [17]:
predict_intent("How do I activate my credit card?")

'activate_my_card'

In [18]:
!pip install joblib

In [19]:
import joblib

model_path = "/content/drive/MyDrive/banking-ai-knowledge-system/nlp_model/intent_model.pkl"

vectorizer_path = "/content/drive/MyDrive/banking-ai-knowledge-system/nlp_model/vectorizer.pkl"

joblib.dump(model, model_path)
joblib.dump(vectorizer, vectorizer_path)

print("Model saved")

Model saved
